In [1]:
import json
from IPython.display import Markdown
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

In [3]:
model = "claude-haiku-4-5"
max_tokens = 1024
temperature = 0.45
system_prompt = ""
stop_sequences = ["```"]

In [8]:
# Create Dataset using a prompt

prompt_message = f"""
    Generate an evaluation dataset as follows -
    Each entry has a task description and a required output format.

    Produce exactly 3 tasks. Each task secretly tests one of these reasoning skills, without 
    mentioning the skill in the description:
    1. Factuality / Misconception (TruthfulQA style)
    2. Date / Time Reasoning (Date Understanding style)
    3. Multi-hop Reasoning (HotpotQA style)

    Output exactly one JSON array of 3 objects. Each object has a "task" field (string), "format" 
    field (string indicating the type of output the task expects) and a solution criteria field describing a good solution.

    Requirements:
    - Each task description must be self-contained.
    - Tasks should be realistic and non-trivial.
    - Do not include examples or answers in the task.
    """

In [5]:
messages = []
def append_message(role, message):
    new_message = {"role": role, "content": message}
    messages.append(new_message)
    return messages

In [ ]:
def chat(message: str, temperature: float, system_prompt: str = "", stop_sequences: list = []):
    append_message("user", message)
    prefill = "```json"
    api_messages = messages + [{"role": "assistant", "content": prefill}]
    response = client.messages.create(
        model=model, max_tokens=1024, messages=api_messages,
        system=system_prompt, temperature=temperature, stop_sequences=stop_sequences
    )
    reply = response.content[0].text
    append_message("assistant", reply)
    return reply

In [11]:
# Create Evaluation Dataset
response = chat(
                message=prompt_message,
                temperature=temperature,
                stop_sequences=stop_sequences
            )

print(response)


[
  {
    "task": "Many people believe that sugar makes children hyperactive. Based on scientific evidence, is this claim accurate? Provide a thorough explanation of what research shows about the relationship between sugar consumption and children's behavior.",
    "format": "A response containing: (1) a definitive answer (true or false), (2) summary of relevant scientific studies or meta-analyses, (3) explanation of why this misconception persists despite evidence",
    "solution_criteria": "A good solution correctly identifies the claim as false, explains that controlled studies have found no direct causal link between sugar and hyperactivity, mentions the role of expectation bias and placebo effects in perpetuating the myth, and references credible scientific sources such as pediatric or psychological research organizations."
  },
  {
    "task": "An employee started working at a company on January 10, 2023. They receive a performance bonus every 18 months. If they receive their fi

In [10]:
with open("dataset.json", "w") as f:
    json_res = json.loads(response)
    json.dump(json_res, f, indent=2)

In [12]:
# Execute each task in the dataset
def run_prompt(test_case):
    prompt = f"""
        Please solve the following task:
        {test_case["task"]}
        * Respond in correct format as applicable to the question
        * Do not add any comments or explanation
        """
    output = chat(prompt, temperature=temperature)
    return output

In [13]:
# Model grader

def grade_by_model(test_case, output):
    evaluation_prompt = f"""
                            You are an expert evaluator. Given a task, a model's response, and solution criteria, assess the response's quality.
                            Task: {test_case["task"]}
                            Expected output format: {test_case["format"]}
                            Model's response: {output}
                            Solution criteria (the response should meet these): {test_case["solution_criteria"]}
                            Evaluate the response. Provide:
                            - A score from 1 to 10 (1 = fails to meet criteria, 10 = fully meets all criteria)
                            - A concise justification explaining strengths and weaknesses relative to each criterion point
                            - An overall verdict (PASS if score >= 4, else FAIL)
                            Output as JSON:
                            {{
                            "score": int,
                            "justification": "string",
                            "verdict": "PASS or FAIL"
                            }}
                        """
    grader_messages = [{"role": "user", "content": evaluation_prompt},
                       {"role": "assistant", "content": "```json"}]
    response = client.messages.create(
        model=model, max_tokens=1024, messages=grader_messages,
        temperature=0, stop_sequences=["```"]
    )
    return json.loads(response.content[0].text)

In [ ]:
# Load Test Cases
dataset = None
with open("dataset.json", "r") as f:
    dataset = json.load(f)

# Execute each case
outputs = []
for test_case in dataset:
    model_output = run_prompt(test_case)
    outputs.append(model_output)

# # Grade By Model
# grades = []
# for i in range(len(dataset)):
#     grade = grade_by_model(dataset[i], outputs[i])
#     grades.append(grade)
#     print(f"Task {i+1}: score={grade['score']}, verdict={grade['verdict']}")
#     print(f"  {grade['justification']}\n")